# Fraud & Suspicious Transaction Pattern Detection


In [ ]:
import pandas as pd
tx=pd.read_csv('../data/processed/transactions_enriched.csv')
alerts=pd.read_csv('../data/processed/fraud_alerts.csv')
risk=pd.read_csv('../data/processed/customer_fraud_risk.csv')
tx.head()


## Fraud Pattern Distribution

In [ ]:
alerts['detected_pattern'].value_counts()


## Channel Exposure

In [ ]:
alerts.groupby('channel')['amount'].agg(['count','sum','mean']).sort_values('sum',ascending=False)


## High-Risk Customers

In [ ]:
risk.sort_values('customer_fraud_score',ascending=False).head(20)


## Confirmed Fraud

In [ ]:
alerts[alerts.disposition=='Confirmed Fraud'].groupby('detected_pattern')['amount'].agg(['count','sum'])


## EDA + Feature Engineering

This section adds a simple exploratory data analysis and feature-engineering workflow to the existing **Fraud & Suspicious Transaction Pattern Detection** project.

The original fraud-pattern distribution, channel exposure, high-risk customer, and confirmed-fraud analysis above is kept unchanged.


### 1. Dataset Review

In [ ]:
# Review the datasets already loaded above
print("Transactions dataset:", tx.shape)
display(tx.head())
display(tx.dtypes.to_frame("data_type"))

print("\nFraud alerts dataset:", alerts.shape)
display(alerts.head())
display(alerts.dtypes.to_frame("data_type"))

print("\nCustomer fraud-risk dataset:", risk.shape)
display(risk.head())
display(risk.dtypes.to_frame("data_type"))


### 2. Missing Values, Duplicates & Data Quality

In [ ]:
# Missing-value and duplicate checks
for name, df in [
    ("Transactions", tx),
    ("Fraud Alerts", alerts),
    ("Customer Fraud Risk", risk)
]:
    print(f"\n{name}")
    print("Duplicate rows:", df.duplicated().sum())

    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    if len(missing):
        display(missing.to_frame("missing_count"))
    else:
        print("No missing values found.")


### 3. Summary Statistics & Range Validation

In [ ]:
# Review important numeric fields already used in the project
for name, df, fields in [
    ("Transactions", tx, ['amount']),
    ("Fraud Alerts", alerts, ['amount']),
    ("Customer Fraud Risk", risk, ['customer_fraud_score'])
]:
    available = [col for col in fields if col in df.columns]

    if available:
        print(f"\n{name} summary")
        display(df[available].describe().T)

# Basic logical checks
if 'amount' in tx.columns:
    print("Negative transaction amounts:", (tx['amount'] < 0).sum())

if 'amount' in alerts.columns:
    print("Negative alert amounts:", (alerts['amount'] < 0).sum())

if 'customer_fraud_score' in risk.columns:
    print("Missing fraud-risk scores:", risk['customer_fraud_score'].isna().sum())


### 4. Simple Outlier Review

In [ ]:
# Simple IQR review of unusually large transaction or alert amounts.
# Fraud-related outliers are flagged for investigation, not automatically removed.
outlier_summary = []

for name, df, col in [
    ("Transactions", tx, "amount"),
    ("Fraud Alerts", alerts, "amount")
]:
    if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        if iqr > 0:
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            count = ((df[col] < lower) | (df[col] > upper)).sum()

            outlier_summary.append({
                "dataset": name,
                "field": col,
                "potential_outliers": int(count)
            })

display(pd.DataFrame(outlier_summary))


### 5. Feature Engineering

These features are intentionally simple and directly connected to fraud and suspicious-transaction analysis.


In [ ]:
# Work on copies so the original project analysis stays unchanged
tx_fe = tx.copy()
alerts_fe = alerts.copy()
risk_fe = risk.copy()

created_tx_features = []
created_alert_features = []
created_risk_features = []

# Transaction amount band
if 'amount' in tx_fe.columns:
    tx_fe['amount_band'] = pd.cut(
        tx_fe['amount'],
        bins=[-float('inf'), 1000, 5000, 10000, float('inf')],
        labels=['Under 1K', '1K-5K', '5K-10K', '10K+']
    )
    created_tx_features.append('amount_band')

    median_amount = tx_fe['amount'].median()
    tx_fe['above_median_amount_flag'] = (tx_fe['amount'] > median_amount).astype(int)
    created_tx_features.append('above_median_amount_flag')

# Alert amount band
if 'amount' in alerts_fe.columns:
    alerts_fe['alert_amount_band'] = pd.cut(
        alerts_fe['amount'],
        bins=[-float('inf'), 1000, 5000, 10000, float('inf')],
        labels=['Under 1K', '1K-5K', '5K-10K', '10K+']
    )
    created_alert_features.append('alert_amount_band')

# Confirmed-fraud flag from the existing disposition field
if 'disposition' in alerts_fe.columns:
    alerts_fe['confirmed_fraud_flag'] = (
        alerts_fe['disposition'].astype(str).str.strip().str.lower() == 'confirmed fraud'
    ).astype(int)
    created_alert_features.append('confirmed_fraud_flag')

# High-risk customer flag using the existing fraud score
if 'customer_fraud_score' in risk_fe.columns:
    high_risk_threshold = risk_fe['customer_fraud_score'].quantile(0.75)
    risk_fe['high_fraud_risk_flag'] = (
        risk_fe['customer_fraud_score'] >= high_risk_threshold
    ).astype(int)
    created_risk_features.append('high_fraud_risk_flag')

print("Transaction features created:", created_tx_features)
print("Alert features created:", created_alert_features)
print("Risk features created:", created_risk_features)

display(tx_fe.head())
display(alerts_fe.head())
display(risk_fe.head())


### 6. Business Rule & KPI Validation

In [ ]:
# Recheck the core project metrics after feature engineering
if 'detected_pattern' in alerts_fe.columns:
    print("Fraud pattern distribution:")
    display(alerts_fe['detected_pattern'].value_counts(dropna=False))

if 'confirmed_fraud_flag' in alerts_fe.columns:
    print("Confirmed fraud cases:", int(alerts_fe['confirmed_fraud_flag'].sum()))
    print("Confirmed fraud rate:",
          round(alerts_fe['confirmed_fraud_flag'].mean() * 100, 2), "%")

if 'channel' in alerts_fe.columns and 'amount' in alerts_fe.columns:
    print("Channel exposure:")
    display(
        alerts_fe.groupby('channel')['amount']
        .agg(['count', 'sum', 'mean'])
        .sort_values('sum', ascending=False)
    )

if 'high_fraud_risk_flag' in risk_fe.columns:
    print("High-risk customers:", int(risk_fe['high_fraud_risk_flag'].sum()))


### 7. Final Validation & Optional Export

In [ ]:
print("Final transaction dataset shape:", tx_fe.shape)
print("Final alert dataset shape:", alerts_fe.shape)
print("Final risk dataset shape:", risk_fe.shape)

print("Transaction duplicates:", tx_fe.duplicated().sum())
print("Alert duplicates:", alerts_fe.duplicated().sum())
print("Risk duplicates:", risk_fe.duplicated().sum())

# Optional exports for Tableau, Streamlit, or further analysis.
# tx_fe.to_csv('../data/processed/transactions_eda_enriched.csv', index=False)
# alerts_fe.to_csv('../data/processed/fraud_alerts_eda_enriched.csv', index=False)
# risk_fe.to_csv('../data/processed/customer_fraud_risk_enriched.csv', index=False)

print("EDA + Feature Engineering completed.")
